In [1]:

import os
import ctypes
import sys
from pathlib import Path
import re

#stuff for circuit -- not needed for file conversion
'''
env_path = '/home/oliviag/.conda/envs/spice_env'
lib_path = os.path.join(env_path, 'lib/libngspice.so')
spice_dir = os.path.join(env_path, 'share/ngspice')
spinit_path = os.path.join(spice_dir, 'scripts/spinit')


os.environ['SPICE_LIB_DIR'] = spice_dir
os.environ['NGSPICE_INPUTINIT'] = spinit_path
os.environ['NGSPICE_LIBRARY_PATH'] = lib_path

from PySpice.Spice.NgSpice.Shared import NgSpiceShared
NgSpiceShared.LIBRARY_PATH = lib_path'''



import numpy as np
import matplotlib.pyplot as plt
from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import *
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed






In [2]:
#functions that may be needed
#checking which files in models dir are not included in a library. May not need.
def missingFileCheck(wrapper):

    with open(wrapper, 'r') as f:
        content = f.read()
    file_list = os.listdir('models')
    notFound = [item for item in file_list if item not in content]
    
    if notFound:
        print(f"{notFound}")


#//////////////////////////////////////////////////////////////////////////////////////////////////
#creating clone model file
def convertModel(modelFile, compFile):
    clean_lines = []
    clean_lines.append("* --- Initializing Dynamic Parameters ---\n")
    #clean_lines.append(".options mode=hspice\n")
    #clean_lines.append(".options chgtol=1e-15\n")
    #clean_lines.append(".param lmin_parm=0 wmin_parm=0 lmax_parm=0 wmax_parm=0\n")
    #clean_lines.append(".param ext_weff1=1u ext_wmob_lc=0 ext_wmob_sc=0 ext_deltau0=0\n\n")
    #clean_lines.append('.param cpp2xflag=0 cpp3xflag=0 enable_extd_vdd=0\n')
    #clean_lines.append('.param l=0 w=0 nf=1 swrfmhc=0 temper=25\n')

    
    if not os.path.exists(modelFile):
        print(f"Error, {modelFile} not found.")
        return
    with open(modelFile, 'r') as f:
        text = f.read()
    text = commentClean(text) 
    #text = text.replace('/home/oliviag', '/home/hweiner/GF22_models') 
    #text = text.replace('fets_ng.lib', 'fets.lib') 
    text = "simulator lang=spectre\n" + text
    text = re.sub(r'^\s*\*', '//', text, flags=re.MULTILINE)
    
    text = re.sub(r'(?i)^\s*\.param\s+', 'parameters ', text, flags=re.MULTILINE)

    text = re.sub(r'(?i)subckt\s+(\w+)\s*\((.*?)\)', r'.subckt \1 \2', text)
    
    text = re.sub(r'(?i)^\s*\.ends\b', 'ends', text, flags=re.MULTILINE)
    text = re.sub(r'=\s*{([^}]+)}', r'=\1', text)
    text = revTernary(text) 
    text = re.sub(r'^\+\s+', '+ ', text, flags=re.MULTILINE)
    


    with open(compFile, 'w') as c: 
        for item in clean_lines:
            c.write(f"{item}")
        c.write("* NGSPICE COMPATIBLE CLONE\n")
        c.write(f"* Source: {modelFile}\n\n")
        c.write(text)
  
    #print(f'Translated to NGSpice: {compFile}')

#//////////////////////////////////////////////////////////////////////////////////////////////////
#changing parameters in model file

def paramUpdate(compFile, param_list):   
    with open(compFile, 'r') as f:
        data = f.read()

    for name, value in paramList.items():
        pattern = rf'^\s*\+?\s*{re.escape(name)}\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)'
        #print(pattern)
        
        if re.search(pattern, data):
            data = re.sub(pattern, lambda m: m.group(1) + str(value), data)
    
    with open(compFile, 'w') as f:
        f.write(data)

#//////////////////////////////////////////////////////////////////////////////////////////////////
#sllowing entire model directory from cadence to be converted with one fucntion call
def process_directory(source_dir: str, dest_dir: str, max_workers: int = 4):
    src_path = Path(source_dir)
    dest_path = Path(dest_dir)
    
    # Identify files in source that need to be processed into destination
    files = [p for p in src_path.rglob("*") if p.is_file()]
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {}
        for f in files:
            # Match the relative structure (e.g., subfolder/file.scs)
            relative_path = f.relative_to(src_path)
            target_file = dest_path / relative_path
            
            # Submit (Source File, Target File to Overwrite)
            futures[executor.submit(convertModel, str(f), str(target_file))] = f

        for future in as_completed(futures):
            file = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"Error updating {file}: {e}")

In [3]:
def revTernary(text):
    # This matches the Ngspice/Hspice logic expansion: 
    # ((cond)*(if_true) + (!(cond))*(if_false))
    # It safely extracts the 3 parts while ignoring internal parentheses.
    pattern = r'\(\s*\(([^)]+)\)\s*\*\s*\(([^)]+)\)\s*\+\s*\(\s*!\s*\(\s*\1\s*\)\s*\)\s*\*\s*\(([^)]+)\)\s*\)'
    
    # We loop to catch nested conditions from inside out
    while re.search(pattern, text):
        # Replaces the math workaround back to standard native Spectre ternary syntax: cond ? if_true : if_false
        text = re.sub(pattern, r'(\1 ? \2 : \3)', text)
        
    # Ensure parameter closing alignments look standard for Spectre
    text = text.replace(' } ', ' }\n')
    return text


In [4]:
#Secondary Functions//////////////////////////////////////////////////////////////////////////////////////////////////
def fixTernary(text): #translating hspice bool to ngspice
    #clean line joins - not needed now
    #text = re.sub(r'\n\s*\+', ' ', text)
    def find_balanced_end(string, start_pos):
        depth = 0
        for i in range(start_pos, len(string)):
            if string[i] == '(': depth += 1
            elif string[i] == ')': depth -= 1
            # If we are at depth 0 and hit a delimiter, we've found the end
            if depth < 0 or (depth == 0 and string[i] in [':', '}', ';']):
                return i
        return len(string)

    while '?' in text:
        q_idx = text.find('?')
        
        # find start of condition
        start_idx = q_idx - 1
        depth = 0
        while start_idx >= 0:
            if text[start_idx] == ')': depth += 1
            elif text[start_idx] == '(': depth -= 1
            if depth < 0 or (depth == 0 and text[start_idx] in ['=', '{', '\n']):
                break
            start_idx -= 1
        
        # find the colon that belongs to this question mark, skip any colons that are inside parentheses
        c_idx = q_idx + 1
        depth = 0
        while c_idx < len(text):
            if text[c_idx] == '(': depth += 1
            elif text[c_idx] == ')': depth -= 1
            if depth == 0 and text[c_idx] == ':':
                break
            c_idx += 1
            
        # find end of false, skip any brackets until the expression ends
        end_idx = find_balanced_end(text, c_idx + 1)

        # extract segments
        cond = text[start_idx+1:q_idx].strip()
        if_true = text[q_idx+1:c_idx].strip()
        if_false = text[c_idx+1:end_idx].strip()

        # build math
        replacement = f"( ({cond})*({if_true}) + (!({cond}))*({if_false}) )"
        
        # update text and continue
        text = text[:start_idx+1] + replacement + text[end_idx:]

    # ensure every param is on one line only
    text = text.replace(' } ', ' }\n')
    return text

#//////////////////////////////////////////////////////////////////////////////////////////////////
def commentClean(text):
    #formatting comments 
    pattern = r'^[ \t]*\*[\*\s]*(.*?)[ \t]*\*?$'
    def handle_match(match):
        content = match.group(1).strip()
            #if line is a definition like '* .param' uncomment and remove space
        if content.startswith('.'):
            return content
        elif content.startswith(' .'):
            return content.lstrip()
            
            # Otherwise, keep it as a clean comment
        return f"* {content}" if content else "*"
    return re.sub(pattern, handle_match, text, flags=re.MULTILINE)
#//////////////////////////////////////////////////////////////////////////////////////////////////


In [5]:
# first 10 params taken from ML repo README, can change them here now instead of manually going into the model file
#can eventually be added to ML pipeline
paramControl = {
    'phig1_nom' : '1.0096807479858398',      
    'u0_nom' : '2.3270835876464844',      
    'eu_nom' : '1.1886792182922363' ,      
    'ucs_nom' : '5.142642021179199' ,      
    'ua_nom' : '0.5721131563186646' ,      
    'ud_nom' : '1.4590672254562378' ,      
    'prwg_nom' : '0.2038799226284027',       
    'cit_nom' : '18.378276824951172' ,      
    'rdw_nom' : '2.6391329765319824',       
    'rsw_nom' : '2.641720771789551' 
    }

#all paths and files are subjective. Alter according to your setup

#converting model file. Using the one in cryoPDK_HSPICE for now (slvnfet). 
modelNG = 'sky130_fd_pr__nfet_01v8_lvt.pm3.spice'
modelSpec = 'spectrePDK.spice'
convertModel(modelNG, modelSpec)

#updating paramaters in model file
#def paramUpdate(modelNG, paramControl)

#converting the fets.lib from cryoPDK_HSPICE setup by hershel
#fetsHspice = 'cryoPDK_HSPICE/fets.lib'
#fetsNG = 'cryoPDK_HSPICE/fets_ng.lib'
#convertModel(fetsHspice, fetsNG)

#converting entire model directory from cadence (files called by fets). 
#2 files will throw an error due to restrictions. this is fine.
#sourcePath = "/home/oliviag/models"
#updatedNgPath = "/home/oliviag/ngSpiceModels/models"
#process_directory(sourcePath, updatedNgPath)





In [8]:
# Test circuit without files
'''circuit = Circuit('Smoke Test')
circuit.V('1', 'n1', circuit.gnd, 5@u_V)
circuit.R('1', 'n1', circuit.gnd, 1@u_kOhm)

print('CHECKPOINT 1')
simulator = circuit.simulator()
print('CHECKPOINT 2')
analysis = simulator.transient(step_time=1@u_ms, end_time=10@u_ms)
print('CHECKPOINT 3 - SUCCESS!')'''

CHECKPOINT 1
CHECKPOINT 2
CHECKPOINT 3 - SUCCESS!


In [14]:
#simulation that does not currently work
'''from PySpice.Spice.NgSpice.Server import SpiceServer
SpiceServer.SPICE_COMMAND = '/home/oliviag/.conda/envs/spice_env/bin/ngspice'

circuit = Circuit('PDK Simulation')
circuit.raw_spice = "set ngbehavior=hsa"



circuit.lib(designWrapNG, 'tt')
circuit.include(modelNG)
circuit.include(fetsNG)
circuit.M('1', 'VPWR', 'GATE', 'VGND', 'VGND', model='slvnfet')
#circuit.M('1', 'slvnfet', 'DRAIN', 'GATE', 'VGND')
#circuit.X('PDK_Sim', 'slvnfet', 'DRAIN', 'GATE', 'VGND', 'VGND')
#circuit.V('gnd', 'VGND', 0, 0)
#circuit.V('dd', 'VPWR', 'VGND', 1.8)
#circuit.R('', 'VPWR', 'DRAIN', '10k')
circuit.V('dd', 'VPWR', 'VGND', 1.8)
circuit.V('in', 'GATE', 'VGND', 0)
print(str(circuit))
print('CHECKPOINT 1')
#simulator = circuit.simulator()
simulator = circuit.simulator(simulator='ngspice-subprocess')
print('CHECKPOINT 2')



analysis = simulator.dc(Vin=slice(0, 1.8, 0.01))
#analysis = simulator.dc(Vin=slice(0, 1.8, 0.1))
#v_gate = analysis.gate     # Or analysis.nodes['gate']
#v_drain = analysis.drain

print('CHECKPOINT 3')

drain_current = -analysis.Vdd 

plt.plot(analysis.sweep, drain_current)
plt.title('MOSFET Id vs Vgs (Transfer Characteristic)')
plt.xlabel('Gate Voltage (V)')
plt.ylabel('Drain Current (A)')
plt.grid(True)
plt.show()'''



'''fig, ax = plt.subplots(figsize=(20, 10))
ax.set_title('mosfet')
ax.set_xlabel('time in 1e-14s')
ax.set_ylabel('voltage in V')
ax.plot(analysis.GATE)
ax.plot(analysis.DRAIN)
ax.legend(('GATE', 'DRAIN'))
plt.tight_layout()
plt.show()'''

.title PDK Simulation
.include /home/oliviag/cryoPDK_HSPICE/cloned_model.inc
.lib /home/oliviag/ngSpiceModels/models/design_wrapper.lib tt
set ngbehavior=hsa
M1 VPWR GATE VGND VGND slvnfet
Vdd VPWR VGND 1.8
Vin GATE VGND 0

CHECKPOINT 1
CHECKPOINT 2


NameError: The number of points was not found in the standard error buffer, ngspice returned:
